# 01 — Data Exploration

Inspect the raw dataset and generate the data-quality report.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

import pandas as pd, numpy as np
import config
from src import data_loader as dl

## Load the raw data

In [ ]:
df = dl.load_raw()
print(df.shape)
df.head()

## Schema and types

In [ ]:
df.info()

## Target and droppable columns

Column roles are *inferred*, not hard-coded.

In [ ]:
target = dl.find_target(df)
drops = dl.detect_droppable(df, target)
print('target:', target)
print('droppable:', drops)

## Missing values and duplicates

In [ ]:
print('missing:')
print(df.isna().sum()[lambda s: s > 0])
print()
print('duplicate rows (excluding ID):', df.drop(columns=list(drops)).duplicated().sum())

## Class balance

In [ ]:
vc = df[target].value_counts()
print(vc)
print()
print('imbalance ratio: %.2f : 1' % (vc.max() / vc.min()))

## The rating-scale observation

Service ratings are documented 1–5 but contain 0. Does 0 behave like 'worst'?

In [ ]:
pos = config.POSITIVE_LABEL
for c in ['In-flight Wifi Service', 'Online Boarding', 'Ease of Online Booking']:
    if c in df.columns:
        rate = df.groupby(c)[target].apply(lambda s: (s == pos).mean())
        print(c)
        print((rate * 100).round(1).to_dict())
        print()

If 0 were the bottom of an ordinal scale these would rise monotonically. They do not — 0 behaves as *not applicable*.

This project keeps ratings as plain 0–5 numeric and documents the finding (see the report).

## Generate the data-quality report

In [ ]:
report = dl.quality_report(df, target, drops)
print(report[:2000])